# 01 — Exploratory Data Analysis (EDA)

## SkinCancerTracker: Multimodal Dermatology Dataset Analysis

This notebook provides a reproducible exploratory analysis of the dermatology dataset using modular utilities from `src.data.eda`.

### Objectives:
1. **Dataset Overview**: Audit row counts, identifier cardinality (`isic_id`, `lesion_id`), images-per-lesion distributions, missing values, and file integrity.
2. **Class Distribution**: Quantify representation across the **11 diagnostic classes** from `training_gt.csv` (`AKIEC`, `BCC`, `BEN_OTH`, `BKL`, `DF`, `INF`, `MAL_OTH`, `MEL`, `NV`, `SCCKA`, `VASC`).
3. **Relational Integrity**: Verify linkage between inputs, ground truth, and clinical metadata without logic duplication.

In [ ]:
# Cell 1: Environment Setup & Modular Imports
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.eda import (
    load_dataset_tables,
    get_dataset_overview,
    get_class_distribution,
    plot_class_distribution,
    plot_images_per_lesion_distribution,
    DIAGNOSTIC_CLASSES_11,
)

print(f"Project root: {project_root}")
print(f"Target 11 diagnostic classes: {DIAGNOSTIC_CLASSES_11}")

In [ ]:
# Cell 2: Load Dataset Tables
data_dir = project_root / "data"
tables = load_dataset_tables(data_dir)

print("Loaded Dataset Tables:")
for key, df in tables.items():
    print(f"  • {key}: {len(df):,} rows x {len(df.columns)} columns")

if not tables:
    print("Note: No local CSVs found in data/. Run validator or provide benchmark CSVs to inspect actual records.")

---  
## A. Dataset Overview

Evaluating dataset cardinality, unique lesions, unique images, and images per lesion.

In [ ]:
# Cell 3: Compute Comprehensive Dataset Overview
image_dir = data_dir / "images"
overview = get_dataset_overview(tables, image_dir=image_dir)

print("=== Row Counts ===")
for name, count in overview.get("row_counts", {}).items():
    print(f"  {name}: {count:,} rows")

print("\n=== Unique Image & Lesion Counts ===")
for name, meta in overview.get("images", {}).items():
    print(f"  {name} images: Unique={meta['unique_isic_ids']}, Dupes={meta['duplicate_isic_ids']}, Nulls={meta['null_isic_ids']}")
for name, meta in overview.get("lesions", {}).items():
    print(f"  {name} lesions: Unique={meta['unique_lesions']}, Dupes={meta['duplicate_lesion_ids']}, Nulls={meta['null_lesion_ids']}")

if "images_per_lesion" in overview:
    ipl = overview["images_per_lesion"]
    print(f"\n=== Images per Lesion Statistics ===")
    print(f"  Min: {ipl['min']}, Max: {ipl['max']}, Mean: {ipl['mean']}, Median: {ipl['median']}")

In [ ]:
# Cell 4: Visualize Images per Lesion Distribution
if "images_per_lesion" in overview:
    fig = plot_images_per_lesion_distribution(overview["images_per_lesion"])
    plt.show()
else:
    print("Skipping plot: No lesion_id data loaded.")

---  
## B. Class Distribution

Analyzing the 11 one-hot diagnostic classes from `training_gt.csv`:
- `AKIEC`, `BCC`, `BEN_OTH`, `BKL`, `DF`, `INF`, `MAL_OTH`, `MEL`, `NV`, `SCCKA`, `VASC`

In [ ]:
# Cell 5: Calculate Diagnostic Class Frequencies
if "training_gt" in tables:
    class_summary = get_class_distribution(tables["training_gt"], class_columns=DIAGNOSTIC_CLASSES_11)
    print("=== 11-Class Diagnostic Distribution Table ===")
    display(class_summary)
else:
    print("training_gt table not loaded. Place training_gt.csv in data/ to calculate class distribution.")

In [ ]:
# Cell 6: Plot Class Distribution Chart
if "training_gt" in tables and not class_summary.empty:
    fig = plot_class_distribution(class_summary)
    plt.show()
else:
    print("Skipping plot: No training_gt data available.")

---  
## C. Summary and Next Steps

- The data validation checks can be run directly from terminal via:
  ```bash
  python -m src.data.validator --data-root data --config configs/data.yaml --strict
  ```
- The findings from this analysis establish lesion cluster structures for patient/lesion-safe group cross-validation splits.